```
# Lab type: debug
# Course: ML301 — Deep Learning with PyTorch
# Lesson: The Training Loop
# Task: The training loop below contains 3 bugs. Each runs without errors but produces wrong results. Find each bug, explain it in the comment cell below it, and fix it.
```

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

# Synthetic binary classification dataset
n_samples = 1000
X = torch.randn(n_samples, 10)
y = (X[:, 0] + X[:, 1] > 0).float()

dataset = TensorDataset(X, y)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f"Dataset: {n_samples} samples, 10 features, binary target")
print(f"Positive rate: {y.mean():.2f}")

Below is a simple feedforward network and a training loop. The loop has 3 bugs — all silent. Read each section carefully before the buggy code is introduced.

In [ ]:
class BinaryClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(10, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        return self.network(x).squeeze(1)

model = BinaryClassifier()
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Bug 1: Gradient accumulation

The optimiser's zero_grad() call is in the wrong position. The loop runs without errors, but gradients accumulate across batches instead of being reset each step.

In [ ]:
# --- BUGGY CODE (Bug 1) ---
model_bug1 = BinaryClassifier()
# Use BCEWithLogitsLoss (correct — no sigmoid needed on output)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_bug1.parameters(), lr=1e-3)

losses_bug1 = []
for epoch in range(3):
    for inputs, targets in train_loader:
        outputs = model_bug1(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()  # BUG: should be BEFORE forward pass, not after step
        losses_bug1.append(loss.item())

print(f"Final loss (buggy): {losses_bug1[-1]:.4f}")
print(f"Loss range: [{min(losses_bug1):.4f}, {max(losses_bug1):.4f}]")
print("Notice: loss is erratic because gradients from previous batches contaminate each update")

**Explain the bug:** Why does calling `optimizer.zero_grad()` after `optimizer.step()` cause incorrect training? What should the correct order be?

*(Write your explanation here.)*

In [ ]:
# Fix for Bug 1: zero_grad() must come BEFORE the forward pass
model_fix1 = BinaryClassifier()
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_fix1.parameters(), lr=1e-3)

losses_fix1 = []
for epoch in range(3):
    for inputs, targets in train_loader:
        optimizer.zero_grad()   # CORRECT: clear before forward
        outputs = model_fix1(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        losses_fix1.append(loss.item())

print(f"Final loss (fixed): {losses_fix1[-1]:.4f}")
print(f"Loss range: [{min(losses_fix1):.4f}, {max(losses_fix1):.4f}]")
print("Smooth, stable gradient updates — loss decreases reliably")

## Bug 2: BCELoss on raw logits

The model outputs raw logits (no sigmoid). `nn.BCELoss` requires probabilities in [0,1] — applying it to logits produces numerically unstable gradients and wrong probability estimates. The code runs without errors.

In [ ]:
# --- BUGGY CODE (Bug 2) ---
model_bug2 = BinaryClassifier()
# BUG: BCELoss requires sigmoid-output probabilities, not raw logits
criterion_bug = nn.BCELoss()
optimizer2 = optim.Adam(model_bug2.parameters(), lr=1e-3)

losses_bug2 = []
for epoch in range(3):
    for inputs, targets in train_loader:
        optimizer2.zero_grad()
        outputs = model_bug2(inputs)   # raw logits, no sigmoid
        loss = criterion_bug(outputs, targets)  # BUG: BCELoss on logits
        loss.backward()
        optimizer2.step()
        losses_bug2.append(loss.item())

print(f"Final loss (buggy BCELoss): {losses_bug2[-1]:.4f}")
print("Warning: BCELoss on logits outside [0,1] produces NaN gradients or wrong results")
print("(Some logit values may cause RuntimeError: NaN loss in stricter configs)")

**Explain the bug:** What does `nn.BCELoss` expect as input, and why is passing raw logits incorrect? What is the numerically stable alternative?

*(Write your explanation here.)*

In [ ]:
# Fix for Bug 2: use BCEWithLogitsLoss which fuses sigmoid + BCE for numerical stability
model_fix2 = BinaryClassifier()
criterion_fix = nn.BCEWithLogitsLoss()  # CORRECT: handles logits directly
optimizer2_fix = optim.Adam(model_fix2.parameters(), lr=1e-3)

losses_fix2 = []
for epoch in range(3):
    for inputs, targets in train_loader:
        optimizer2_fix.zero_grad()
        outputs = model_fix2(inputs)
        loss = criterion_fix(outputs, targets)
        loss.backward()
        optimizer2_fix.step()
        losses_fix2.append(loss.item())

print(f"Final loss (fixed): {losses_fix2[-1]:.4f}")
print("BCEWithLogitsLoss = log-sum-exp trick internally — stable and correct")

## Bug 3: scheduler.step() inside the batch loop

A learning rate scheduler is supposed to adjust the LR once per epoch. Calling it inside the batch loop reduces the LR on every batch — with 32 batches per epoch and 3 epochs, the LR decays 96 times instead of 3.

In [ ]:
# --- BUGGY CODE (Bug 3) ---
model_bug3 = BinaryClassifier()
criterion3 = nn.BCEWithLogitsLoss()
optimizer3 = optim.Adam(model_bug3.parameters(), lr=1e-3)
# StepLR: multiply LR by gamma every step_size steps
scheduler = optim.lr_scheduler.StepLR(optimizer3, step_size=1, gamma=0.5)

lr_history = []
for epoch in range(3):
    for inputs, targets in train_loader:
        optimizer3.zero_grad()
        outputs = model_bug3(inputs)
        loss = criterion3(outputs, targets)
        loss.backward()
        optimizer3.step()
        scheduler.step()  # BUG: should be outside the batch loop (once per epoch)
        lr_history.append(optimizer3.param_groups[0]['lr'])

print(f"LR at start: {lr_history[0]:.2e}")
print(f"LR at batch 10: {lr_history[9]:.2e}")
print(f"LR at end: {lr_history[-1]:.2e}")
print(f"LR decayed {len(lr_history)} times — should have decayed {3} times (once per epoch)")

**Explain the bug:** How many times does the scheduler step in the buggy version vs the correct version? What is the practical effect on the learning rate?

*(Write your explanation here.)*

In [ ]:
# Fix for Bug 3: scheduler.step() called once per epoch, after all batches
model_fix3 = BinaryClassifier()
criterion3_fix = nn.BCEWithLogitsLoss()
optimizer3_fix = optim.Adam(model_fix3.parameters(), lr=1e-3)
scheduler_fix = optim.lr_scheduler.StepLR(optimizer3_fix, step_size=1, gamma=0.5)

lr_history_fix = []
for epoch in range(3):
    for inputs, targets in train_loader:
        optimizer3_fix.zero_grad()
        outputs = model_fix3(inputs)
        loss = criterion3_fix(outputs, targets)
        loss.backward()
        optimizer3_fix.step()
    scheduler_fix.step()  # CORRECT: once per epoch
    lr_history_fix.append(optimizer3_fix.param_groups[0]['lr'])

print(f"LR after epoch 1: {lr_history_fix[0]:.2e}")
print(f"LR after epoch 2: {lr_history_fix[1]:.2e}")
print(f"LR after epoch 3: {lr_history_fix[2]:.2e}")
print(f"LR decayed {len(lr_history_fix)} times — correct (once per epoch)")

## Summary

> **For each bug, write one sentence summarising what went wrong and how to prevent it.**

1. 
2. 
3. 